# **Часть первая - Наполнение векторной базы**

In [ ]:
# Установка библиотеки (если ещё не установлена)
#!pip install python-docx

In [ ]:
#pip install --upgrade transformers

In [1]:
PINECONE_API_KEY = "pcsk_2ghPWB_3v4p9rrdZ85hp2SzJnMMEPELbpE1cPiXhXUaiBQueTq7C3Wy1ag271dGJnbbdG7" # акк Анны

#vectorizing_model = 'all-MiniLM-L6-v2'
vectorizing_model = 'sentence-transformers/distilbert-base-nli-mean-tokens'

index_name = 'vac-res-match' # "red-llama-bertopic"
index_dimension=768 #384

### Извлечение текста из вордовских файлов

In [2]:
from docx import Document
import os

def extract_text_from_docx(folder_path, file_number):
    """
    Извлекает текст из файла {file_number}.docx в указанной папке.
    Обрабатывает как параграфы, так и текст внутри таблиц.

    Args:
        folder_path (str): Путь к папке с файлами (например, r"C:\CV")
        file_number (int): Номер файла (1 до 65)

    Returns:
        str: Извлечённый текст из .docx файла или пустая строка, если файл не найден/ошибка
    """
    file_name = f"{file_number}.docx"
    file_path = os.path.join(folder_path, file_name)

    try:
        doc = Document(file_path)
        full_text = []

        # 1. Извлекаем текст из параграфов
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():  # Только непустые параграфы
                full_text.append(paragraph.text.strip())

        # 2. Извлекаем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        if paragraph.text.strip():  # Только непустые параграфы в ячейках
                            full_text.append(paragraph.text.strip())

        # Объединяем все тексты с разрывами строк
        return '\n'.join(full_text)

    except FileNotFoundError:
        print(f"⚠️ Файл не найден: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Ошибка при чтении {file_path}: {e}")
        return ""


In [3]:
# === Обработка резюме в цикле extract_text_from_docx по файлам 1..65 ===

# Путь к папке CV
CV_FOLDER_PATH = r"vacancy-resume-matching-dataset-main\CV"

# Проверим, существует ли папка
if not os.path.exists(CV_FOLDER_PATH):
    print(f"❌ Папка не существует: {CV_FOLDER_PATH}")
    print("Убедитесь, что путь указан правильно. Возможно, нужно использовать абсолютный путь.")
else:
    print(f"✅ Папка найдена: {CV_FOLDER_PATH}\n")

    # Собираем тексты всех файлов в список
    texts = []
    for i in range(1, 66):  # от 1 до 65 включительно
        print(f"📄 Обработка файла {i}.docx...")
        text = extract_text_from_docx(CV_FOLDER_PATH, i)
        texts.append(text)  # сохраняем для дальнейшего использования, если нужно
    
    print("\n=== Готово! ===")
    print(f"Обработано файлов: {len([t for t in texts if t])} из 65")


✅ Папка найдена: vacancy-resume-matching-dataset-main\CV

📄 Обработка файла 1.docx...
📄 Обработка файла 2.docx...
📄 Обработка файла 3.docx...
📄 Обработка файла 4.docx...
📄 Обработка файла 5.docx...
📄 Обработка файла 6.docx...
📄 Обработка файла 7.docx...
📄 Обработка файла 8.docx...
📄 Обработка файла 9.docx...
📄 Обработка файла 10.docx...
📄 Обработка файла 11.docx...
📄 Обработка файла 12.docx...
📄 Обработка файла 13.docx...
📄 Обработка файла 14.docx...
📄 Обработка файла 15.docx...
📄 Обработка файла 16.docx...
📄 Обработка файла 17.docx...
📄 Обработка файла 18.docx...
📄 Обработка файла 19.docx...
📄 Обработка файла 20.docx...
📄 Обработка файла 21.docx...
📄 Обработка файла 22.docx...
📄 Обработка файла 23.docx...
📄 Обработка файла 24.docx...
📄 Обработка файла 25.docx...
📄 Обработка файла 26.docx...
📄 Обработка файла 27.docx...
📄 Обработка файла 28.docx...
📄 Обработка файла 29.docx...
📄 Обработка файла 30.docx...
📄 Обработка файла 31.docx...
📄 Обработка файла 32.docx...
📄 Обработка файла 33.do

In [4]:
# Проверить, что текст первого файла не пустой
if texts[0]:
    print("Текст первого резюме:\n", texts[0][:500], "...")


Текст первого резюме:
 ****************
Java full stack developer
(60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB FRONTEND)
3+ years of experience FullStack development and 5+ total experience in software development
phone**********	 		linkedIn
location Bat Yam     		email
Professional Skills
Programming languages: Java, JavaScript.
Web:  HTML5/CSS3, JQuery, Bootstrap3
Environments – IDE/tools:  Eclipse, IntelliJ IDEA, GitHub.
Databases: SQL (MySQL), NoSQL (MongoDB).
Technologies & Frameworks: OOP, AOP, Spring MVC ...


In [5]:
def print_text_by_file_number(texts, file_number):
    """
    Распечатывает текст из списка `texts`, соответствующий номеру файла.
    
    Args:
        texts (list): Список строк, где texts[i] — текст из файла (i+1).docx
        file_number (int): Номер файла (1–65)
    """
    if not isinstance(file_number, int) or file_number < 1 or file_number > 65:
        print(f"❌ Неверный номер файла: {file_number}. Допустимы числа от 1 до 65.")
        return
    
    index = file_number - 1  # Преобразуем номер файла в индекс списка (0-based)
    
    if not texts[index]:  # Проверяем, не пустой ли текст
        print(f"📄 Файл {file_number}.docx: **текст отсутствует или не удалось прочитать**")
    else:
        print(f"\n{'='*60}")
        print(f"📄 ТЕКСТ ИЗ ФАЙЛА {file_number}.docx")
        print(f"{'='*60}")
        print(texts[index])
        print(f"{'='*60}\n")


In [6]:
# === Пример использования ===

# Пример: распечатать текст из файла 5.docx
print_text_by_file_number(texts, 12)

# Пример: распечатать текст из файла 1
print_text_by_file_number(texts, 1)

# Пример: попробовать неверный номер
print_text_by_file_number(texts, 100)


📄 ТЕКСТ ИЗ ФАЙЛА 12.docx
Haifa, Israel      Languages   Personal qualities Quick learner Flexibility Communication skills Problem solving Time management Leadership Critical thinking   Education Computing machines, systems, complexes, and networks            2006 – 2012  Moscow Aviation Institute (National Research University) Honours degree
Front End developer Summary A senior web developer with 10+ years development experience. Created both the web and the mobile applications utilizing JavaScript, Typescript, React. Have full-Stack development experience and deep knowledge of the databases (SQL, PostgreSQL). Participated in more than 10 successful projects. Implemented several projects from the scratch (combined the roles of a developer and a project/product manager), and in other projects led the development teams.  Experience Front End Team lead / Senior Developer                               2020 – Present AviaBit Performed a developer role while leading the web and mobile appli

### ✅ Код: Подготовка `my_chunks` — один чанк = один документ (резюме)

In [7]:

my_chunks = []

for i, text in enumerate(texts, start=1):  # i от 1 до 65
    if text.strip():  # пропускаем пустые тексты
        chunk = {
                "chunk_text": text,       # полный текст резюме
                "source": f"{i}.docx" # исходный файл
        }
        my_chunks.append(chunk)
    else:
        print(f"⚠️ Пропущен пустой файл: {i}.docx")

print(f"✅ Готово! Подготовлено {len(my_chunks)} чанков (документов) для загрузки в Pinecone.")

✅ Готово! Подготовлено 65 чанков (документов) для загрузки в Pinecone.


In [8]:
# Проверка чанка с индексом chunk_index, начинается с 0
chunk_index = 11
if 0 <= chunk_index < len(my_chunks):
    chunk = my_chunks[chunk_index]
    print(f"=== ЧАНК №{chunk_index + 1} ===")
    print(f"File: {chunk['source']}")
    print(f"Текст:\n{chunk['chunk_text']}")
    print("=" * 60)
else:
    print(f"❌ Чанк с индексом {chunk_index} не существует. Всего чанков: {len(my_chunks)}")


=== ЧАНК №12 ===
File: 12.docx
Текст:
Haifa, Israel      Languages   Personal qualities Quick learner Flexibility Communication skills Problem solving Time management Leadership Critical thinking   Education Computing machines, systems, complexes, and networks            2006 – 2012  Moscow Aviation Institute (National Research University) Honours degree
Front End developer Summary A senior web developer with 10+ years development experience. Created both the web and the mobile applications utilizing JavaScript, Typescript, React. Have full-Stack development experience and deep knowledge of the databases (SQL, PostgreSQL). Participated in more than 10 successful projects. Implemented several projects from the scratch (combined the roles of a developer and a project/product manager), and in other projects led the development teams.  Experience Front End Team lead / Senior Developer                               2020 – Present AviaBit Performed a developer role while leading the web and 

# Загрузка на Pinecone

In [9]:
import transformers
from sentence_transformers import SentenceTransformer

print("transformers version:", transformers.__version__)
model = SentenceTransformer(vectorizing_model)
print("✅ SentenceTransformer загружен успешно!")


transformers version: 4.57.6
✅ SentenceTransformer загружен успешно!


In [10]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# Загрузка модели
# model = SentenceTransformer(vectorizing_model)

# Векторизация чанков
chunk_embeddings = model.encode(my_chunks)

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY) 

# Создание индекса

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=index_dimension,  # Размерность эмбеддингов
        metric="cosine",  # Метрика для поиска
        spec=ServerlessSpec(
            cloud="aws",  # Облачный провайдер
            region="us-east-1"  # Регион
        )
    )

# Подключение к индексу
index = pc.Index(index_name)

# Сохранение чанков в Pinecone
for i, (chunk, embedding) in enumerate(zip(my_chunks, chunk_embeddings)):
    index.upsert([(f"chunk_{i}", embedding, chunk)])

print("Чанки успешно сохранены в Pinecone!")


Чанки успешно сохранены в Pinecone!


# **Часть вторая - Проверка и получение чанков с Pinecone**

### 🔧 Шаг 1: Инициализация Pinecone и загрузка модели

In [12]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Загрузка модели
model = SentenceTransformer(vectorizing_model)

print(f"✅ Модель '{vectorizing_model}' загружена. Размерность вектора: {model.get_sentence_embedding_dimension()}")


✅ Модель 'sentence-transformers/distilbert-base-nli-mean-tokens' загружена. Размерность вектора: 768


### 🧠 Шаг 2: Функция для генерации эмбеддинга текста


In [11]:
def get_embedding(text):
    """
    Генерирует эмбеддинг текста с помощью model.
    """
    # Модель ожидает строку или список строк
    embedding = model.encode(text, convert_to_numpy=False).tolist()
    return embedding
    

### 🔍 Шаг 3: Поиск топ-3 чанков по запросу

In [12]:
def search_top_k_chunks(query_text, top_k=3):
    """
    Ищет top-k наиболее релевантных чанков по косинусной близости.
    Возвращает список словарей с метаданными и оценкой сходства.
    """
    # 1. Получаем эмбеддинг запроса
    query_vector = get_embedding(query_text)

    # 2. Запрашиваем топ-k результатов из Pinecone
    response = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,   # ⚠️ КЛЮЧЕВОЕ: чтобы получить текст и метаданные
        include_values=False     # Не нужны сами векторы
    )

    # 3. Извлекаем результаты
    results = []
    for match in response.matches:
        results.append({
            "id": match.id,
            "score": match.score,           # косинусная близость (0..1)
            "text": match.metadata.get("chunk_text", "[отсутствует]"),
            "file": match.metadata.get("source", "неизвестно"),
        })

    return results
    

### 🚀 Шаг 4: Использование — пример поиска

### Извлечение вакансии

In [13]:
vac_num = 3 # Номер вакансии в дата-сете

In [14]:
import pandas as pd
import os

# Путь к папке с датасетом (должен быть в той же директории, что и скрипт)
DATASET_PATH = "vacancy-resume-matching-dataset-main"

# Полный путь к файлу
csv_file = os.path.join(DATASET_PATH, "5_vacancies.csv")

# Проверяем, существует ли файл
if not os.path.exists(csv_file):
    raise FileNotFoundError(f"Файл не найден: {csv_file}\nУбедитесь, что папка 'vacancy-resume-matching-dataset-main' находится в текущей директории.")

# Загружаем CSV
df_vacancies = pd.read_csv(csv_file)

# Проверяем, что столбец 'job_description' существует
if 'job_description' not in df_vacancies.columns:
    raise ValueError("В файле нет столбца 'job_description'. Проверьте структуру CSV.")



# Берём запись с номером vac_num
job_description = df_vacancies.iloc[vac_num]['job_description']

# Выводим результат
print(f"🎯 Извлечённая job_description из строки {vac_num} 5_vacancies.csv:\n")
print("="*80)
print(job_description)
print("="*80)

# Сохраняем в переменную user_query для дальнейшего использования
user_query = job_description

# Пример: можно сразу использовать в поиске (если у вас есть Pinecone)
print("\n✅ Готово! Теперь переменная 'user_query' содержит текст вакансии для поиска.")


🎯 Извлечённая job_description из строки 3 5_vacancies.csv:

Minimum Required Skills: LAMP, Drupal, ElasticSearch, REST APIs, Python, Go, AWS, View.js, HubSpot, RESTful APIs Our company is a rapidly growing ratings and reviews platform headquartered in Washington D.C. We are the leading service covering Business-to-Business (B2B) services firms and are looking for teammates to help us accelerate the growth of our platform. Top Reasons to Work with Us 1. Ranked on the LinkedIn Top Companies | Startups list! 2. Highly competitive base pay and benefits 3. World renown leadership and room to grow professionally 4. Top 50 US Startup What You Will Be Doing We emphasize professional growth so responsibilities will evolve over time based on the company ' s needs and the candidate ' s skills/interests. Initial responsibilities will include the following: - Developing new features - Implementing a microservices-based architecture - Providing operational support (when stuff breaks, you help fix it

### Поиск в векторной базе по косинусному сходству с вакансией


In [15]:
# Ваш текстовый запрос
# user_query = ""

# Ищем топ-3 чанка
top3 = search_top_k_chunks(user_query, top_k=3)

print(f"🔍 Поиск по запросу: «{user_query}»\n")
print("🏆 ТОП-3 релевантных чанка:\n")

for i, result in enumerate(top3, 1):
    print(f"{i}. 📄 ID: {result['id']}")
    print(f"   🔍 Сходство: {result['score']:.4f}")
    print(f"   📁 Файл: {result['file']}")
    print(f"   📝 Текст: {result['text'][:200]}...")
    print("-" * 80)


🔍 Поиск по запросу: «Minimum Required Skills: LAMP, Drupal, ElasticSearch, REST APIs, Python, Go, AWS, View.js, HubSpot, RESTful APIs Our company is a rapidly growing ratings and reviews platform headquartered in Washington D.C. We are the leading service covering Business-to-Business (B2B) services firms and are looking for teammates to help us accelerate the growth of our platform. Top Reasons to Work with Us 1. Ranked on the LinkedIn Top Companies | Startups list! 2. Highly competitive base pay and benefits 3. World renown leadership and room to grow professionally 4. Top 50 US Startup What You Will Be Doing We emphasize professional growth so responsibilities will evolve over time based on the company ' s needs and the candidate ' s skills/interests. Initial responsibilities will include the following: - Developing new features - Implementing a microservices-based architecture - Providing operational support (when stuff breaks, you help fix it) - Maintaining and developing Drupal c

### Запрос по ID

In [16]:
# Запрос по ID
chunk_id = "chunk_0"
response = index.fetch(ids=[chunk_id])

    
if chunk_id in response.vectors:
    metadata = response.vectors[chunk_id].metadata
    text = metadata.get("chunk_text", "Текст не найден в метаданных")
    file = metadata.get("source", "неизвестно")

    print(f"📄 Чанк ID: {chunk_id}")
    print(f"📁 Файл: {file}")
    print(f"📝 Текст: {text[:300]}{'...' if len(text) > 300 else ''}")
else:
    print(f"❌ Чанк с ID '{chunk_id}' не найден в индексе.")


📄 Чанк ID: chunk_0
📁 Файл: 1.docx
📝 Текст: ****************
Java full stack developer
(60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB FRONTEND)
3+ years of experience FullStack development and 5+ total experience in software development
phone**********	 		linkedIn
location Bat Yam     		email
Professional Skills
Programming languages: Ja...


### Получить статистику индекса (количество векторов)

In [17]:
# Получить статистику индекса (количество векторов)
stats = index.describe_index_stats()
total_vectors = stats["total_vector_count"]
print(f"Всего чанков в индексе: {total_vectors}")

Всего чанков в индексе: 65
